# 🇻🇳 GHÉ THĂM 63 TỈNH THÀNH VIỆT NAM — TIKTOK LIVE INTERACTIVE GAME 9:16
### HỆ THỐNG PHÁT SÓNG GAME TƯƠNG TÁC BẢN ĐỒ VIỆT NAM TRÊN GOOGLE COLAB

Game tương tác trực tiếp theo chuẩn khung hình dọc **9:16 (1080 x 1920 px)** cho TikTok Live:
- **Tương tác thông minh:** Nhận diện cả 34 đơn vị hành chính cấp tỉnh mới (theo Nghị quyết 2025) và 63 tỉnh thành truyền thống, tự động zoom camera vào ranh giới tỉnh kèm giọng đọc AI (TTS) tiếng Việt hài hước.
- **Đoàn Đồng Hương & Bầu Trưởng Đoàn:** Khi nhiều khán giả cùng chọn một tỉnh, hệ thống tự động gom vào 1 đoàn, phong Trưởng đoàn 👑 cho người donate nhiều xu nhất và cộng điểm ưu tiên quy mô đoàn (+20 điểm/thành viên).
- **Nhập đoàn tức thì (Instant Drop-in):** Khán giả bình luận tỉnh đang chiếu trên màn hình sẽ bay vào nhập đoàn ngay lập tức kèm hiệu ứng và giọng đọc chào mừng.
- **Bảng Theo Dõi Bình Luận & Biểu Đồ Đua Top Quê Hương:** Cập nhật thời gian thực trên màn hình game và trang Quản trị Admin.
- **Bộ Kết Nối TikTok Live Hiện Đại:** Hỗ trợ đổi tài khoản hoặc kết nối lại ngay trên trang Quản trị Admin (`/admin.html`) mà không cần khởi động lại Colab.
- **2 Chế độ phát trên Google Colab:**
  - **Mode A (Khuyên dùng):** Tạo đường link HTTPS Cloudflare Tunnel dán thẳng vào mục **Browser Source** (1080 x 1920) trong **TikTok LIVE Studio** hoặc **OBS Studio**.
  - **Mode B (Tự động 24/7 - Rảnh tay):** Tự tạo Màn hình ảo (Xvfb 1080x1920) và dùng FFmpeg đẩy luồng RTMP thẳng lên TikTok bằng **Stream Key** mà không cần mở máy tính cá nhân!

In [ ]:
# @title 🚀 MODE A: Khởi Chạy Server & Lấy Link Dán Vào TikTok LIVE Studio / OBS { display-mode: "form" }
# @markdown Điền tên tài khoản TikTok (ví dụ: `your_username` hoặc `@your_username`) và các thông số ban đầu rồi bấm nút **Play ▶** bên trái:

TikTok_Username = "vuatiengviet171095" # @param {type:"string"}
Muc_Tieu_Follower = 10000 # @param {type:"integer"}
Follower_Hien_Tai = 0 # @param {type:"integer"}
Tich_Luy_Nguoi_Xem_Ban_Dau = 0 # @param {type:"integer"}

import os, sys, time, re, subprocess
from IPython.display import display, HTML

print("⏳ [1/4] Đang đồng bộ mã nguồn Game Ghé Thăm 63 Tỉnh Thành...")
# Nếu chạy trên Colab và chưa có thư mục app thì clone về
!git clone https://github.com/tuna1710/tiktok-live-game-skill.git /content/app 2>/dev/null || (cd /content/app && git fetch origin && git reset --hard origin/main && git pull 2>/dev/null || true)

# Tìm thư mục game
game_dir = "/content/app/games/vietnam-tour-live"
if not os.path.exists(game_dir):
    game_dir = "/content/vietnam-tour-live"
if not os.path.exists(game_dir):
    game_dir = os.getcwd()

os.chdir(game_dir)
print(f"📁 Thư mục làm việc: {game_dir}")

print("⏳ [2/4] Đang kiểm tra thư viện Node.js & Cloudflare Tunnel...")
!npm install --silent
![ ! -f /usr/local/bin/cloudflared ] && wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

print("⏳ [3/4] Đang dọn dẹp tiến trình cũ và khởi chạy Game Server...")
!pkill -9 -f "server.js" > /dev/null 2>&1
!pkill -9 -f "cloudflared" > /dev/null 2>&1
time.sleep(1)

clean_username = TikTok_Username.replace('@', '').strip()
env = os.environ.copy()
env["PORT"] = "3000"
if clean_username:
    env["TIKTOK_USERNAME"] = clean_username
env["FOLLOWERS_TARGET"] = str(Muc_Tieu_Follower)
env["FOLLOWERS_START"] = str(Follower_Hien_Tai)
env["TOTAL_VIEWERS_START"] = str(Tich_Luy_Nguoi_Xem_Ban_Dau)

with open("/tmp/game_server.log", "w") as f_log:
    server_proc = subprocess.Popen(["node", "server.js"], cwd=game_dir, env=env, stdout=f_log, stderr=subprocess.STDOUT)
time.sleep(3)

print("⏳ [4/4] Đang tạo đường truyền HTTPS công khai qua Cloudflare Tunnel...")
with open("/tmp/cloudflared.log", "w") as cf_log:
    tunnel_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:3000"], stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(25):
    time.sleep(1)
    if os.path.exists("/tmp/cloudflared.log"):
        with open("/tmp/cloudflared.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    obs_link = f"{public_url}/index.html"
    admin_link = f"{public_url}/admin.html"
    html_card = f"""
    <div style="background: radial-gradient(circle at 50% 20%, #0d2735 0%, #030a11 100%); border: 2px solid #ffe600; border-radius: 16px; padding: 22px; color: #ffffff; font-family: sans-serif; box-shadow: 0 8px 30px rgba(255,230,0,0.3);">
        <h2 style="color: #ffe600; margin-top: 0; font-size: 24px;">🇻🇳 GAME GHÉ THĂM 63 TỈNH THÀNH ĐÃ SẴN SÀNG!</h2>
        <p style="font-size: 14.5px; line-height: 1.6; color: #e0e6ed;">
            Đường link công khai HTTPS đã được thiết lập qua Cloudflare Tunnel.<br>
            Tài khoản TikTok Live đang kết nối: <b style="color: #00f2fe;">@{clean_username or 'Chế độ chờ/Mock'}</b>
        </p>
        <div style="margin: 18px 0; display: flex; flex-wrap: wrap; gap: 12px;">
            <a href="{obs_link}" target="_blank" style="background: linear-gradient(135deg, #ffe600 0%, #ff9900 100%); color: #000; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px; box-shadow: 0 4px 15px rgba(255,230,0,0.4);">
                📺 [1] MỞ MÀN HÌNH GAME (Dán link này vào OBS / TikTok Studio)
            </a>
            <a href="{admin_link}" target="_blank" style="background: linear-gradient(135deg, #00b894 0%, #00cec9 100%); color: #000; padding: 13px 22px; border-radius: 9px; text-decoration: none; font-weight: bold; font-size: 15px;">
                🎛️ [2] TRANG QUẢN TRỊ & KẾT NỐI TIKTOK (ADMIN PANEL)
            </a>
        </div>
        <div style="background: rgba(0,0,0,0.6); border-left: 4px solid #ffe600; padding: 12px 14px; border-radius: 6px; font-size: 13.5px; color: #ffe4b5; margin-bottom: 10px;">
            📌 <b>Link dán vào OBS / TikTok Studio Browser Source (Kích thước: 1080 x 1920 px):</b><br>
            <code style="color: #00f2fe; background: #000; padding: 5px 10px; border-radius: 4px; display: inline-block; margin-top: 6px; font-size: 14px; word-break: break-all;">{obs_link}</code>
        </div>
        <div style="background: rgba(0,242,254,0.1); border-left: 4px solid #00f2fe; padding: 10px 14px; border-radius: 6px; font-size: 13px; color: #c4f1f9;">
            💡 <b>Mẹo tiện lợi:</b> Bạn có thể đổi tài khoản streamer hoặc kết nối lại bất cứ lúc nào ngay trong <b>Trang Quản Trị [2]</b> mà không cần khởi chạy lại Colab!
        </div>
    </div>
    """
    display(HTML(html_card))
else:
    print("❌ Chưa lấy được link Cloudflare. Log chi tiết:")
    !cat /tmp/cloudflared.log


---
### 🔴 HOẶC CHẠY CHẾ ĐỘ PHÁT NGẦM RTMP (KHÔNG CẦN OBS)

In [ ]:
# @title 🔴 MODE B: Phát Trực Tiếp Lên TikTok Bằng Stream Key (Không Cần OBS / Máy Tính) { display-mode: "form" }
# @markdown Lấy URL máy chủ và Khóa luồng (Stream Key) từ TikTok LIVE Studio -> Dán vào đây -> Bấm nút **Play ▶**:

Stream_URL = "rtmp://live-push.tiktok.com/live/" # @param {type:"string"}
Stream_Key = "" # @param {type:"string"}
TikTok_Username = "vuatiengviet171095" # @param {type:"string"}

!python3 live_ngam_tiktok.py --stream-url "$Stream_URL" --stream-key "$Stream_Key" --tiktok-username "$TikTok_Username"
